# GemAI — RAG Evaluation

---

### Objective

Evaluate the GemAI RAG pipeline using a hand-crafted test dataset and LangChain evaluators. We will:

1. Load and chunk the same PDFs used by the app
2. Create a hand-crafted evaluation dataset with questions verified against the source PDFs
3. Run a baseline evaluation
4. Run experiments to improve retrieval and answer quality
5. Compare results and document findings

## Part 0: Setup

In [ ]:
# Uncomment if needed
# !pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 langchain-qdrant==0.2.0
# !pip install -qU langgraph==0.2.61 unstructured==0.16.12 pymupdf>=1.24
# !pip install -qU langchain>=0.3.14 langchain-core>=0.3.28 langchain-text-splitters>=0.3.0
# !pip install -qU langsmith>=0.2.0 nltk>=3.9 qdrant-client>=1.12
# !pip install -qU langchain-experimental>=0.3.0

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"GemAI-Eval-{uuid4().hex[0:8]}"
print(f"LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")

LangSmith Project: GemAI-Eval-e0393e72


## Part 1: Load Documents

In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader("../data/", glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Loaded {len(docs)} document pages")

Loaded 61 document pages


/home/celine/AIM/notebooks/graded-mini-projects/14/.venv/lib/python3.11/site-packages/langchain_community/document_loaders/parsers/pdf.py:322: UserWarning: Warning: Empty content on page 19 of document ../data/usgs-natural-gemstones.pdf
  warnings.warn(


## Part 2: Evaluation Dataset

Hand-crafted test questions based on the gemstone corpus, with reference answers verified against the source PDFs.

In [5]:
import pandas as pd

test_data = [
    {
        "question": "What is the hardness of diamond on the Mohs scale?",
        "reference": "Diamond has a hardness of 10 on the Mohs scale, making it the hardest natural substance."
    },
    {
        "question": "Where were sapphires first found in Montana?",
        "reference": "Sapphires in Judith Basin County, Montana, were first found when the gravels were worked for gold from 1895 to 1930."
    },
    {
        "question": "What is the Hope Diamond?",
        "reference": "The Hope Diamond is a famous large blue diamond in the Smithsonian Institution's National Gem Collection, donated by Harry Winston, Inc."
    },
    {
        "question": "What makes a mineral qualify as a gemstone?",
        "reference": "A gemstone must have beauty, durability, and rarity. A precious gemstone has all three qualities, while a semiprecious gemstone has only one or two."
    },
    {
        "question": "What is specific gravity and how is it used in gemology?",
        "reference": "Specific gravity is the ratio of a gem's weight to the weight of an equal volume of water. It helps distinguish between gems that look similar."
    },
    {
        "question": "How did the Smithsonian gem collection begin?",
        "reference": "The National Gem Collection began in 1884 when Prof. F. W. Clarke prepared an exhibit of American precious stones for the Smithsonian Institution's display at the New Orleans Exposition."
    },
    {
        "question": "What are the main types of gem cuts?",
        "reference": "The main types of gem cuts include faceted cuts (like brilliant and step cuts), cabochon cuts (smooth rounded tops), and tumbled stones."
    },
    {
        "question": "What is the difference between a natural and synthetic gemstone?",
        "reference": "A natural gemstone forms in nature, while a synthetic gemstone is created in a laboratory. Synthetics have the same chemical and physical properties as their natural counterparts."
    },
    {
        "question": "What gemstones are found in pegmatite deposits?",
        "reference": "Pegmatite deposits commonly contain tourmaline, beryl (including emerald and aquamarine), and many other gem-quality minerals."
    },
    {
        "question": "What was the combined value of US gemstone production in 2025?",
        "reference": "The combined value of U.S. natural and synthetic gemstone output in 2025 was an estimated $47 million, a decrease of 30% compared with 2024."
    },
]

df = pd.DataFrame(test_data)
print(f"Test dataset: {len(df)} examples")
df

Test dataset: 10 examples


,question,reference
0,What is the hardness of diamond on the Mohs sc...,Diamond has a hardness of 10 on the Mohs scale...
1,Where were sapphires first found in Montana?,"Sapphires in Judith Basin County, Montana, wer..."
2,What is the Hope Diamond?,The Hope Diamond is a famous large blue diamon...
3,What makes a mineral qualify as a gemstone?,"A gemstone must have beauty, durability, and r..."
4,What is specific gravity and how is it used in...,Specific gravity is the ratio of a gem's weigh...
5,How did the Smithsonian gem collection begin?,The National Gem Collection began in 1884 when...
6,What are the main types of gem cuts?,The main types of gem cuts include faceted cut...
7,What is the difference between a natural and s...,"A natural gemstone forms in nature, while a sy..."
8,What gemstones are found in pegmatite deposits?,Pegmatite deposits commonly contain tourmaline...
9,What was the combined value of US gemstone pro...,The combined value of U.S. natural and synthet...


In [6]:
from langsmith import Client

client = Client()
dataset_name = f"GemAI Eval - {os.environ['LANGCHAIN_PROJECT']}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="GemAI Gemstone RAG Evaluation"
)

for _, row in df.iterrows():
    client.create_example(
        inputs={"question": row["question"]},
        outputs={"answer": row["reference"]},
        dataset_id=langsmith_dataset.id
    )

print(f"Dataset '{dataset_name}' created with {len(df)} examples")

Dataset 'GemAI Eval - GemAI-Eval-e0393e72' created with 10 examples


## Part 3: Evaluators

In [8]:
from langchain_openai import ChatOpenAI
from langsmith.evaluation import LangChainStringEvaluator, evaluate

eval_llm = ChatOpenAI(model="gpt-4.1")

qa_evaluator = LangChainStringEvaluator("qa", config={"llm": eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm": eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

groundedness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "groundedness": (
                "Is this response grounded in factual information? "
                "Does it avoid making claims not supported by the source material?"
            ),
        },
        "llm": eval_llm
    }
)

evaluators = [qa_evaluator, labeled_helpfulness_evaluator, groundedness_evaluator]
print("Evaluators ready: QA Correctness, Labeled Helpfulness, Groundedness")

Evaluators ready: QA Correctness, Labeled Helpfulness, Groundedness


## Part 4: Baseline Pipeline

Matches the app's current configuration:

| Parameter | Baseline Value |
|-----------|---------------|
| Chunk Size | 800 |
| Chunk Overlap | 100 |
| Embedding | text-embedding-3-small |
| Retrieval k | 4 |
| LLM | gpt-4o-mini |
| Prompt | GemAI system prompt |

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
baseline_documents = text_splitter.split_documents(docs)
print(f"Baseline: {len(baseline_documents)} chunks (chunk_size=800, overlap=100)")

Baseline: 205 chunks (chunk_size=800, overlap=100)


In [10]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant

baseline_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
baseline_vectorstore = Qdrant.from_documents(
    documents=baseline_documents,
    embedding=baseline_embeddings,
    location=":memory:",
    collection_name="baseline"
)
baseline_retriever = baseline_vectorstore.as_retriever(search_kwargs={"k": 4})

In [11]:
from langchain.prompts import ChatPromptTemplate

BASELINE_PROMPT = """\
You are GemAI, a friendly and knowledgeable assistant specializing in gemstones.
Your knowledge comes from USGS gemstone publications and the Smithsonian
Institution gem collection reference.

Based on the following context, answer the user's question accurately.
If the context does not contain the answer, say so directly rather than guessing.
Keep answers clear, concise, and educational.
Never use profanity or inappropriate language.

Context: {context}
Question: {question}
"""

baseline_prompt = ChatPromptTemplate.from_template(BASELINE_PROMPT)

In [12]:
from operator import itemgetter
from langchain.schema import StrOutputParser

baseline_llm = ChatOpenAI(model="gpt-4o-mini")

baseline_chain = (
    {"context": itemgetter("question") | baseline_retriever, "question": itemgetter("question")}
    | baseline_prompt | baseline_llm | StrOutputParser()
)

# Quick test
print(baseline_chain.invoke({"question": "What is the hardness of diamond?"}))

Diamond is rated as 10 on the Mohs scale of hardness, making it the hardest known substance.


### Baseline Evaluation

In [13]:
baseline_results = evaluate(
    baseline_chain.invoke,
    data=dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "baseline"},
)
print("\n=== BASELINE RESULTS ===")
print(baseline_results)

/home/celine/AIM/notebooks/graded-mini-projects/14/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'terrific-popcorn-25' at:
https://smith.langchain.com/o/52420107-7b88-4243-acba-2f92a36f9bc4/datasets/d3ae206c-fbc2-46d3-8614-eb14832addc9/compare?selectedSessions=09982982-eed4-4fc3-8aa7-d03f3a10cf67




10it [01:27,  8.77s/it]


=== BASELINE RESULTS ===
<ExperimentResults terrific-popcorn-25>


### Baseline Scores

| Metric | Baseline Score |
|--------|---------------|
| QA Correctness | 0.8 |
| Labeled Helpfulness | 0.8 |
| Groundedness | 0.7 |

---

## Part 5: Experiments

### Experiment 1: Chunk Size (800 → 1200)

**Hypothesis:** Larger chunks will provide more surrounding context per retrieval hit, improving QA Correctness and Helpfulness. Gemstone fact sheets have structured paragraphs that may get split at 800 chars.

In [14]:
# ============================================================
# EXPERIMENT 1: Chunk Size 800 -> 1200
# ============================================================

text_splitter_exp1 = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=100)
exp1_documents = text_splitter_exp1.split_documents(docs)
print(f"Experiment 1: {len(exp1_documents)} chunks (chunk_size=1200, overlap=100)")

exp1_vectorstore = Qdrant.from_documents(
    documents=exp1_documents,
    embedding=baseline_embeddings,
    location=":memory:",
    collection_name="experiment-1"
)
exp1_retriever = exp1_vectorstore.as_retriever(search_kwargs={"k": 4})

exp1_chain = (
    {"context": itemgetter("question") | exp1_retriever, "question": itemgetter("question")}
    | baseline_prompt | baseline_llm | StrOutputParser()
)

print(exp1_chain.invoke({"question": "What is the hardness of diamond?"}))

Experiment 1: 146 chunks (chunk_size=1200, overlap=100)
The hardness of diamond is rated as 10 on the Mohs scale, making it the hardest known substance.


In [15]:
exp1_results = evaluate(
    exp1_chain.invoke,
    data=dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "exp1_chunk1200"},
)
print("\n=== EXPERIMENT 1 RESULTS ===")
print(exp1_results)

View the evaluation results for experiment: 'memorable-heart-98' at:
https://smith.langchain.com/o/52420107-7b88-4243-acba-2f92a36f9bc4/datasets/d3ae206c-fbc2-46d3-8614-eb14832addc9/compare?selectedSessions=7612c2d7-9a1d-4349-a8bc-62fcfd9b6584




10it [01:00,  6.09s/it]


=== EXPERIMENT 1 RESULTS ===
<ExperimentResults memorable-heart-98>


**Experiment 1 Results:**

| Metric | Baseline | Experiment 1 | Change (Δ) |
|--------|----------|-------------|------------|
| QA Correctness | 0.8 | 0.8 | No change |
| Labeled Helpfulness | 0.8 | 0.8 | No change |
| Groundedness | 0.7 | 0.8 | + 0.1 |

**Analysis:**

Increasing chunk size from 800 to 1200 had a modest effect — QA Correctness and Helpfulness stayed flat, while Groundedness improved by 0.1. The chunk count dropped from 205 to 146, meaning each chunk carries more surrounding context. This likely helped Groundedness because the LLM had more complete paragraphs to draw from, reducing the need to infer or fill gaps. However, the lack of improvement in Correctness suggests the baseline 800-char chunks were already splitting at reasonable boundaries for our structured gemstone corpus. The trade-off is that larger chunks use more of the context window per retrieval hit, which could become a concern at higher topK values.

---

### Experiment 2: Retrieval k (4 → 8)

**Hypothesis:** Increasing topK from 4 to 8 will retrieve more relevant context, improving QA Correctness — especially for questions that require information spread across multiple sections of the corpus.

In [16]:
# ============================================================
# EXPERIMENT 2: Retrieval k = 4 -> 8
# ============================================================

exp2_vectorstore = Qdrant.from_documents(
    documents=baseline_documents,
    embedding=baseline_embeddings,
    location=":memory:",
    collection_name="experiment-2"
)
exp2_retriever = exp2_vectorstore.as_retriever(search_kwargs={"k": 8})

exp2_chain = (
    {"context": itemgetter("question") | exp2_retriever, "question": itemgetter("question")}
    | baseline_prompt | baseline_llm | StrOutputParser()
)

print(exp2_chain.invoke({"question": "What is the hardness of diamond?"}))

The hardness of diamond is rated at 10 on the Mohs scale, making it the hardest known substance.


In [17]:
exp2_results = evaluate(
    exp2_chain.invoke,
    data=dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "exp2_topk8"},
)
print("\n=== EXPERIMENT 2 RESULTS ===")
print(exp2_results)

View the evaluation results for experiment: 'sparkling-dog-38' at:
https://smith.langchain.com/o/52420107-7b88-4243-acba-2f92a36f9bc4/datasets/d3ae206c-fbc2-46d3-8614-eb14832addc9/compare?selectedSessions=f6ea53fd-bd8c-478e-9474-e5cba34388cd




10it [01:11,  7.13s/it]


=== EXPERIMENT 2 RESULTS ===
<ExperimentResults sparkling-dog-38>


**Experiment 2 Results:**

| Metric | Baseline | Experiment 2 | Change (Δ) |
|--------|----------|-------------|------------|
| QA Correctness | 0.8 | 0.9 | + 0.1 |
| Labeled Helpfulness | 0.8 | 0.8 | No change |
| Groundedness | 0.7 | 0.8 | + 0.1 |

**Analysis:**

Doubling topK from 4 to 8 produced the best overall improvement among the three experiments. QA Correctness rose to 0.9 and Groundedness improved to 0.8, while Helpfulness held steady. The Correctness gain confirms the hypothesis — some questions require information spread across multiple chunks (e.g., gemstone occurrences mentioned in both the Smithsonian and USGS sources), and retrieving more chunks increases the chance of capturing all relevant pieces. The Groundedness improvement likely stems from the same effect: with more source material in the context, the LLM has less reason to go beyond what was retrieved. Helpfulness staying flat suggests that 4 chunks already provided enough context for clear, useful answers — the extra chunks helped accuracy without changing the overall quality of the response style.

---

### Experiment 3: Embedding Model (text-embedding-3-small → text-embedding-3-large)

**Hypothesis:** A larger embedding model should produce higher-quality vector representations, improving semantic matching and retrieval precision, which should improve QA Correctness.

In [18]:
# ============================================================
# EXPERIMENT 3: Embedding model upgrade
# ============================================================

exp3_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
exp3_vectorstore = Qdrant.from_documents(
    documents=baseline_documents,
    embedding=exp3_embeddings,
    location=":memory:",
    collection_name="experiment-3"
)
exp3_retriever = exp3_vectorstore.as_retriever(search_kwargs={"k": 4})

exp3_chain = (
    {"context": itemgetter("question") | exp3_retriever, "question": itemgetter("question")}
    | baseline_prompt | baseline_llm | StrOutputParser()
)

print(exp3_chain.invoke({"question": "What is the hardness of diamond?"}))

The hardness of diamond is rated as 10 on the Mohs scale, making it the hardest known substance.


In [19]:
exp3_results = evaluate(
    exp3_chain.invoke,
    data=dataset_name,
    evaluators=evaluators,
    metadata={"revision_id": "exp3_large_embedding"},
)
print("\n=== EXPERIMENT 3 RESULTS ===")
print(exp3_results)

View the evaluation results for experiment: 'flowery-pot-78' at:
https://smith.langchain.com/o/52420107-7b88-4243-acba-2f92a36f9bc4/datasets/d3ae206c-fbc2-46d3-8614-eb14832addc9/compare?selectedSessions=6ad40166-37d6-4bbe-9dfc-3f00a9fc8849




10it [01:10,  7.01s/it]


=== EXPERIMENT 3 RESULTS ===
<ExperimentResults flowery-pot-78>


**Experiment 3 Results:**

| Metric | Baseline | Experiment 3 | Change (Δ) |
|--------|----------|-------------|------------|
| QA Correctness | 0.8 | 0.9 | + 0.1 |
| Labeled Helpfulness | 0.8 | 0.7 | - 0.1 |
| Groundedness | 0.7 | 0.8 | + 0.1 |

**Analysis:**

Upgrading the embedding model improved QA Correctness (+0.1) and Groundedness (+0.1), confirming that `text-embedding-3-large` produces better semantic representations that retrieve more relevant chunks. However, Helpfulness dropped by 0.1 — an unexpected result. One possible explanation: the larger embedding model retrieves chunks that are more *precisely* relevant but potentially narrower in scope, leading to answers that are technically correct but less comprehensive. With the smaller model, slightly "fuzzier" matches may have pulled in neighboring context that made answers feel more helpful. This is an interesting trade-off — better retrieval precision doesn't always translate to better user experience. The higher embedding cost of `text-embedding-3-large` also needs to be weighed against marginal gains.

---

## Part 6: Final Comparison & Recommendation

### Comparison Table

| Metric | Baseline | Exp 1 (Chunk 1200) | Exp 2 (TopK 8) | Exp 3 (Large Embed) |
|--------|----------|---------------------|-----------------|----------------------|
| QA Correctness | 0.8 | 0.8 | **0.9** | **0.9** |
| Labeled Helpfulness | 0.8 | 0.8 | **0.8** | 0.7 |
| Groundedness | 0.7 | 0.8 | **0.8** | 0.8 |
| **revision_id** | baseline | exp1_chunk1200 | exp2_topk8 | exp3_large_embedding |

### Best Configuration

**My recommended configuration is:** Experiment 2 — TopK 8 (with baseline chunk size 800 and `text-embedding-3-small`)

**Justification:**

Experiment 2 (topK=8) is the clear winner. It achieved the highest QA Correctness (0.9) while being the only experiment that maintained Helpfulness at the baseline level (0.8) and improved Groundedness (0.8). In contrast, Experiment 3 matched Correctness but *decreased* Helpfulness, and Experiment 1 showed no Correctness improvement at all.

Increasing topK is also the cheapest change to implement — it requires no re-embedding or re-chunking, just a single parameter change in the retrieval call. The cost trade-off is minimal: doubling the retrieved chunks slightly increases the prompt token count sent to gpt-4o-mini, but at the model's low per-token cost this is negligible.

For the deployed GemAI app, I recommend changing topK from 4 to 8 in `app/api/chat/route.ts`.